[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/04_%E5%9B%A0%E6%9E%9C%E6%8E%A8%E8%AB%96_%E5%82%BE%E5%90%91%E3%82%B9%E3%82%B3%E3%82%A2%E3%81%A8IPW.ipynb)

In [ ]:
# === ① セットアップ（最初に実行）: 教材用の合成データを作ります ===
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False
rng = np.random.default_rng(11)
n = 800
関心度 = rng.normal(0, 1, n)                                   # 観測できる交絡（既存の購買意欲）
p_treat = 1/(1+np.exp(-(0.3 + 1.3*関心度)))                    # 意欲が高い企業ほどウェビナーに来やすい
ウェビナー = (rng.random(n) < p_treat).astype(int)             # 施策＝ウェビナー参加(1)/不参加(0)
lin = -1.1 + 1.4*関心度 + 0.6*ウェビナー                       # 受注のlogit（関心度もウェビナーも効く）
受注 = (rng.random(n) < 1/(1+np.exp(-lin))).astype(int)
df = pd.DataFrame({'関心度': 関心度.round(2), 'ウェビナー': ウェビナー, '受注': 受注})
# 種明かし用の『真の効果』(ATE)：全員参加 と 全員不参加 の受注確率の差
p1 = 1/(1+np.exp(-(-1.1+1.4*関心度+0.6))); p0 = 1/(1+np.exp(-(-1.1+1.4*関心度)))
TRUE_ATE = (p1 - p0).mean()
print('準備OK（教材用の合成データ。真の効果はあとで答え合わせします）')

# 発展マーケ-04. 因果推論（傾向スコア・IPW）

> 🟢 Colab（ブラウザ）で実行できます。**最初に「① セットアップ」セルを実行**してください。
> 📌 **発展編（統計検定2級の先：準1級〜実務／データサイエンス）**。statsmodels・scipy を使います（Colabは導入済み。ローカルは `uv add statsmodels scipy`）。

**舞台設定**：ウェビナーに参加した企業は受注率が高い。でも「ウェビナーのおかげ」？ それとも「もともと買う気のある企業が参加しただけ」？ 観察データから**因果効果**を取り出します。

**使う統計（読む=準1級／本質=1級）**：傾向スコア・逆確率重み付け(IPW)・平均処置効果(ATE)。

### 📋 使うデータ：教材用の合成データ `df`（企業800件）
`関心度`＝既存の購買意欲（交絡）、`ウェビナー`＝施策に参加したか(1/0)、`受注`＝成約したか(1/0)。

> ⚠️ 因果推論は「真の効果」と答え合わせしたいので、効果を仕込んだ合成データを使います。

## 🎯 この章でできるようになること
- 観察データの単純比較がなぜ **交絡** で歪むかを説明できる
- **傾向スコア**（処置を受ける確率）を推定できる
- **IPW（逆確率重み付け）** で交絡を調整した効果（ATE）を出せる

> **前提**：ロジスティック回帰（発展01）　/　**所要**：約30分　/　**レベル**：発展（読む準1級・本質1級）

## 1. 単純比較の罠（交絡）

まず素朴に「参加群」と「不参加群」の受注率を比べます。この差には **ウェビナーの効果＋もともとの意欲の差** が混ざっています。

In [ ]:
naive = df.groupby('ウェビナー')['受注'].mean()
print(naive)
naive_diff = naive[1] - naive[0]
print(f'\n素朴な差（参加 − 不参加）= {naive_diff:+.3f}')
# 参加群のほうが関心度も高い → これが交絡
print('参加群/不参加群の平均関心度:', df.groupby('ウェビナー')['関心度'].mean().round(2).to_dict())

## 2. 傾向スコア：処置を受ける確率を推定

**傾向スコア** e(x)=P(参加 | 関心度) を、関心度からロジスティック回帰で推定します。「同じくらい参加しやすい企業どうし」を比べる準備です。

In [ ]:
import statsmodels.formula.api as smf
ps_model = smf.logit('ウェビナー ~ 関心度', data=df).fit(disp=0)
df['ps'] = ps_model.predict(df)            # 傾向スコア e(x)
fig, ax = plt.subplots(figsize=(7,4))
for t,lab in [(1,'参加'),(0,'不参加')]:
    ax.hist(df[df['ウェビナー']==t]['ps'], bins=25, alpha=.6, label=lab)
ax.set_xlabel('傾向スコア e(x)=P(参加|関心度)'); ax.set_ylabel('企業数'); ax.legend()
ax.set_title('両群のスコアが重なっている＝比較可能(positivity)'); plt.show()

## 3. IPW（逆確率重み付け）で調整

「参加しにくいのに参加した企業」「参加しやすいのに不参加だった企業」を重く数え直すと、両群の関心度の偏りが打ち消され、**疑似的にランダム化**した状態になります。

- 参加群の重み = 1/e(x)　/　不参加群の重み = 1/(1−e(x))

In [ ]:
t = df['ウェビナー'].values; y = df['受注'].values; e = df['ps'].values
w1 = t/e; w0 = (1-t)/(1-e)                  # 逆確率重み
mu1 = (w1*y).sum()/w1.sum()                 # 参加していたらの受注率（重み付き）
mu0 = (w0*y).sum()/w0.sum()                 # 不参加だったらの受注率（重み付き）
ate_ipw = mu1 - mu0
print(f'IPW調整後のATE = {ate_ipw:+.3f}')

## 4. 答え合わせ：素朴な差 vs IPW vs 真の効果

In [ ]:
print(f'素朴な差     : {naive_diff:+.3f}  ← 交絡で過大評価')
print(f'IPW調整後    : {ate_ipw:+.3f}  ← 交絡を調整')
print(f'真の効果(ATE): {TRUE_ATE:+.3f}  ← 種明かし')
print('\n→ 素朴な差は真の効果より大きい。IPWは真の効果に近づく。')

## 🧭 まとめ
- 観察データの単純比較は **交絡** で歪む（意欲の高い企業が施策に集まる、など）。
- **傾向スコア**で「処置の受けやすさ」をそろえ、**IPW**で重み付けすると交絡を調整できる。
- 調整後の **ATE** は、施策の純粋な効果に近づく。

> ⚠️ **よくある間違い**
> - **未観測の交絡**はIPWでも消えない（測っていない要因で偏っていたらアウト）。これが因果推論の難所。
> - 傾向スコアが0や1に近い企業がいると重みが爆発する（**positivity**の崩れ）。両群スコアの重なりを必ず確認。
> - 「相関が出た＝因果」ではない。無作為化実験(RCT)か、こうした調整が必要。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: 傾向スコア e=0.25 の『参加群』の重み 1/e を ans に
ans = None   # 例: 1/0.25
_check('Q1 参加群の重み', ans, 1/0.25)

In [ ]:
# Q2: 傾向スコア e=0.20 の『不参加群』の重み 1/(1−e) を ans に
ans = None   # 例: 1/(1-0.20)
_check('Q2 不参加群の重み', ans, 1/(1-0.20))

In [ ]:
# Q3: 重み付き受注率が 参加=0.42, 不参加=0.30 のとき ATE=参加−不参加 を ans に
ans = None   # 例: 0.42-0.30
_check('Q3 ATE', ans, 0.42-0.30)

---
## 🏆 練習問題

**問1.** `関心度` を傾向スコアに入れない（`ウェビナー ~ 1` の定数モデル）と、IPWの結果は素朴な差に戻ることを確かめよう。なぜ？

**問2.** 重み `w1`,`w0` の最大値を見て、極端に大きい重み（positivityの崩れ）が無いか確認しよう。

**問3.**（考察）「未観測交絡」の例を、BtoBマーケの文脈で1つ挙げよう（IPWでは調整できない要因）。

> 🔑 **解答例は別ページ**（ネタバレ防止）👉 **[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/04_%E5%9B%A0%E6%9E%9C%E6%8E%A8%E8%AB%96_%E5%82%BE%E5%90%91%E3%82%B9%E3%82%B3%E3%82%A2%E3%81%A8IPW.md)**
>
> 自分で解いてから開きましょう。

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 交絡 | 処置と結果の両方に効く第3の要因 |
| 傾向スコア e(x) | 処置を受ける確率 P(T=1\|x) |
| IPW | 1/e で重み付けし交絡を調整 |
| ATE | 平均処置効果（全体での効果） |
| positivity | どの層にも処置/非処置が居る前提 |

```python
e = smf.logit('T ~ x', data=df).fit().predict(df)
mu1 = (df.T/e*df.Y).sum()/(df.T/e).sum()
mu0 = ((1-df.T)/(1-e)*df.Y).sum()/((1-df.T)/(1-e)).sum()
ate = mu1 - mu0
```